# 13.03 · Building an LLM API

**Chapter:** Serving & Deployment · **Milestone:** M5

## Goals
- [ ] Wrap an LLM app in a **FastAPI** service with Pydantic-validated requests and responses
- [ ] Stream tokens with **Server-Sent Events (SSE)** and measure time-to-first-token
- [ ] Add API-key auth, rate limiting, timeouts and clean error handling
- [ ] Understand why `async` matters for LLM apps (you spend most of the time waiting on the network)
- [ ] Test the service without starting a server (`TestClient`) and with one (`uvicorn`)

## Resources
- [FastAPI docs](https://fastapi.tiangolo.com/) (tutorial + "Advanced User Guide") — free
- [MDN – Server-sent events](https://developer.mozilla.org/en-US/docs/Web/API/Server-sent_events/Using_server-sent_events) — free
- [OpenAI – Streaming responses](https://platform.openai.com/docs/guides/streaming-responses) — free
- Alternatives: [LitServe](https://github.com/Lightning-AI/LitServe), [BentoML](https://docs.bentoml.com/), [Ray Serve](https://docs.ray.io/en/latest/serve/) — free/OSS

Your notebook prototype becomes a product when other programs can call it: a web front-end, a mobile app, a Slack bot, another team's service.
The API is the contract: what goes in, what comes out, who may call it, how often, and what happens when things fail.

The service lives in **`app.py`** next to this notebook (open it side by side). It serves the M3 RAG pipeline (`aieng.rag`) with:
`GET /healthz`, `POST /v1/ask` (JSON answer + sources) and `POST /v1/ask/stream` (SSE tokens).

In [ ]:
import asyncio
import json
import os
import atexit
import subprocess
import sys
import time

import httpx
from fastapi.testclient import TestClient

import app as service

print(open("app.py").read()[:1200], "...")

## 1. Contract first: Pydantic models
`AskRequest` rejects bad input *before* any code runs (question length 3–500 chars, `k` between 1 and 10): FastAPI returns **422** with a
precise error. `AskResponse` documents the output and appears in the auto-generated OpenAPI docs at `/docs`.
Limits on input size are also a security and cost control: nobody can send you a 2 MB "question".

## 2. Test in-process with `TestClient`
No server, no ports: requests go straight into the app. `with` runs the startup hook (loads the RAG index).

In [ ]:
H = {"Authorization": "Bearer dev-key"}
with TestClient(service.app) as tc:
    print("health      ", tc.get("/healthz").json())
    print("no key      ", tc.post("/v1/ask", json={"question": "Who was Lincoln?"}).status_code)
    bad = tc.post("/v1/ask", json={"question": "hi", "k": 50}, headers=H)
    print("bad input   ", bad.status_code, [e["loc"][-1] + ": " + e["msg"] for e in bad.json()["detail"]])
    r = tc.post("/v1/ask", json={"question": "Who was Abraham Lincoln?", "k": 3}, headers=H)
    print("ok          ", r.status_code, r.json()["latency_ms"], "ms")
    print(r.json()["answer"][:300])
    print("sources:", [s["id"] for s in r.json()["sources"]])

## 3. Rate limiting
LLM calls cost money, so every public endpoint needs a per-key limit. `app.py` uses a sliding window in memory (fine for one process;
use Redis or your gateway's limiter (11.02) when you run several). Over the limit → **429** with `Retry-After`.

In [ ]:
service.RATE_PER_MIN, codes = 3, []
service._hits.clear()
with TestClient(service.app) as tc:
    for _ in range(5):
        r = tc.post("/v1/ask", json={"question": "x" * 600}, headers=H)  # invalid on purpose: no LLM call, but auth+limit still run
        codes.append(r.status_code)
print(codes, "-> 422 = passed the limiter (then failed validation), 429 = limited")
service.RATE_PER_MIN = 20

Order matters: here validation of the body happens after the dependency, so even invalid requests consume quota (good: it stops abuse).

## 4. Run it for real and stream with SSE
SSE is plain HTTP: the response stays open and the server writes `event: …\ndata: …\n\n` blocks. Browsers support it natively (`EventSource`),
it passes through proxies, and it's what OpenAI-style APIs use. (WebSockets are for two-way realtime, e.g. voice in 14.03.)

In [ ]:
server = subprocess.Popen([sys.executable, "-m", "uvicorn", "app:app", "--port", "8765", "--log-level", "warning"])
atexit.register(server.terminate)  # stop it even if a later cell fails
for _ in range(120):
    try:
        httpx.get("http://localhost:8765/healthz"); break
    except httpx.ConnectError:
        time.sleep(0.5)
print("server up:", httpx.get("http://localhost:8765/healthz").json())

In [ ]:
def ask_stream(question):
    start, ttft, text = time.perf_counter(), None, ""
    with httpx.stream("POST", "http://localhost:8765/v1/ask/stream", json={"question": question}, headers=H, timeout=60) as r:
        event = None
        for line in r.iter_lines():
            if line.startswith("event: "):
                event = line[7:]
            elif line.startswith("data: "):
                data = json.loads(line[6:])
                if event == "sources":
                    print("sources:", [s["id"] for s in data])
                elif event == "token":
                    ttft = ttft or time.perf_counter() - start
                    text += data
                elif event == "error":
                    print("stream error:", data)
    print(text[:400])
    print(f"\nTTFT {ttft * 1000 if ttft else float('nan'):.0f} ms | total {(time.perf_counter() - start) * 1000:.0f} ms")


ask_stream("What did Abraham Lincoln do before becoming president?")

Streaming doesn't make the answer faster: it makes the *first words* arrive sooner, which is what users perceive. Always report TTFT and total.
Some providers (Gemini) send the answer in a few large chunks, so TTFT sits close to total; Ollama streams token by token.
Try `APP_PROVIDER=ollama` or `APP_PROVIDER=groq` and compare.

## 5. Why async: concurrent requests
The handler `await`s the LLM, so while one request waits on the network the event loop serves others. CPU work (embedding the query) runs in
a thread via `asyncio.to_thread` so it doesn't freeze the loop. Five concurrent requests take about as long as the slowest one, not the sum.

In [ ]:
async def burst(n):
    async with httpx.AsyncClient(timeout=60) as ac:
        async def one(i):
            t = time.perf_counter()
            r = await ac.post("http://localhost:8765/v1/ask", json={"question": f"Who was Abraham Lincoln? ({i})", "k": 2}, headers=H)
            return r.status_code, time.perf_counter() - t
        t = time.perf_counter()
        results = await asyncio.gather(*(one(i) for i in range(n)))
        return results, time.perf_counter() - t


results, wall = await burst(5)
print("status:", [s for s, _ in results])
print(f"sum of latencies {sum(t for _, t in results):.1f}s vs wall clock {wall:.1f}s")

In [ ]:
server.terminate()
_ = server.wait()

A blocking call (e.g. `requests.post(...)` or the sync OpenAI client) inside an `async def` handler would serialise everything: the classic FastAPI bug.
Either use async clients, or declare the handler with plain `def` (FastAPI then runs it in a thread pool).

## 6. Production checklist
- **Timeouts everywhere**: client → your API → provider (`AsyncOpenAI(timeout=30, max_retries=1)`); fail fast and return 502/504, not a hung socket.
- **Errors**: don't leak provider messages or stack traces; log them with a **request ID** and return a generic message.
- **Auth**: API keys hashed in a DB, or OAuth/JWT for user-facing apps; secrets only in env vars / a secret manager.
- **Observability**: trace every request (11.01), log tokens and cost per key, expose latency percentiles (13.06).
- **Workers**: `uvicorn app:app --workers 4` (or gunicorn with uvicorn workers) uses all CPU cores; each worker has its own memory (and rate limiter).
- **CORS** (`fastapi.middleware.cors`) if a browser front-end on another domain calls you.
- **Versioning**: `/v1/...` so you can change the contract later without breaking clients.

## Exercises
1. Add `POST /v1/chat/completions` that accepts the OpenAI chat format and answers with RAG. Point Open WebUI or the OpenAI SDK at your API.
2. Add a per-key **daily token budget** (reuse `Budget` from 11.02) that returns 402 when exceeded.
3. Add tracing with Phoenix (11.01) so each API request shows retrieval and LLM spans.
4. Write `test_app.py` with pytest + `TestClient` covering 401, 422, 429 and a mocked LLM (monkeypatch `service.llm`) so it runs in CI without keys.

In [ ]:
# your code here